In [1]:
import gffpandas.gffpandas as gffpd
import pandas as pd
import numpy as np
from Bio import Entrez, SeqIO 
from urllib.request import urlopen
from io import StringIO
import re
from intervaltree import Interval, IntervalTree

In [2]:
from Bio import Entrez, SeqIO

In [3]:
ctrl_ph7 = gffpd.read_gff3('../sample1_modification.gff') #Wild Type pH 7
arss_ph7 = gffpd.read_gff3('../sample2_modification.gff') #ArsS deletion pH 7, cannot detect pH change
#hsdm_repair_pH7 = gffpd.read_gff3('sample3_modification.gff') #hsdM deletion and repair pH 7, test if pseudogene
hsdm_ph7 = gffpd.read_gff3('../sample4_modification.gff') #hsdM deletion pH 7, cannot methylated all motifs
ctrl_ph5 = gffpd.read_gff3('../sample5_modification.gff') #wild type pH 5
arss_ph5 = gffpd.read_gff3('../sample6_modification.gff') #ArsS deletion pH 5
#hsdm_repair_ph5 = gffpd.read_gff3('sample7_modification.gff') #hsdM deletion and repair pH 5, test if pseudogene
hsdm_ph5 = gffpd.read_gff3('../sample8_modification.gff') #hsdM deletion and repair

In [4]:
ctrl_ph7 = ctrl_ph7.attributes_to_columns()
ctrl_ph5 = ctrl_ph5.attributes_to_columns()

arss_ph7 = arss_ph7.attributes_to_columns()
arss_ph5 = arss_ph5.attributes_to_columns()

hsdm_ph7 = hsdm_ph7.attributes_to_columns()
hsdm_ph5 = hsdm_ph5.attributes_to_columns()

In [5]:
Entrez.email = "skpatterson@wm.edu"

handle = Entrez.efetch(db="nucleotide", id="CP003904.1", rettype="fasta", retmode="text")
fasta_string = handle.read()
handle.close()

In [6]:
print(fasta_string[0:100])

>CP003904.1 Helicobacter pylori 26695, complete genome
TGATTAGTGATTAGTGATTAGTGATTAGTGATTAGTGATTAGTGA


In [7]:
# alternate sequencing option for CP003904 fasta
Entrez.email = "skpatterson@wm.edu"

handle = Entrez.efetch(db="nucleotide", id="NC_018939.1", rettype="fasta", retmode="text")
fasta_string = handle.read()
handle.close()

In [8]:
fasta_io = StringIO(fasta_string)
record = SeqIO.read(fasta_io, "fasta")
sequence = str(record.seq)

In [9]:
sequence[3]

'T'

# CGANNNNNNTC Motif Indexing

In [20]:
def find_motifs(sequence): # for 
    pattern = 'CGA......TC'  #equivalent to CGANNNNNNTC
    results = []

    for match in re.finditer(pattern, sequence):
        start = match.start()
        end = match.end()
        actual_motif = sequence[start:end]
        results.append({
            'Start': start,
            'End': end,
            'Motif': actual_motif
        })

    return pd.DataFrame(results)

# Assuming you've already fetched and stored your sequence as a string variable 'sequence'
cga = find_motifs(sequence)

In [21]:
cga.head()

,Start,End,Motif
0,3680,3691,CGAAATGTATC
1,5752,5763,CGATCCCATTC
2,8759,8770,CGATGAGTTTC
3,10440,10451,CGATAACGATC
4,17458,17469,CGAAGCTTATC


In [22]:
ctrl_ph7[ctrl_ph7['start'] <= 1714]

,seq_id,source,type,start,end,score,strand,phase,attributes,IPDRatio,context,coverage,id,identificationQv,motif
2,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,255,255,713,+,.,context=AGTGAGCTTTTTGCTCAAAGAATCCAAGATAGCGTTTA...,5.35,AGTGAGCTTTTTGCTCAAAGAATCCAAGATAGCGTTTAAAA,534,GANTC,656,GANTC
5,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,modified_base,279,279,53,+,.,coverage=552;context=CAAGATAGCGTTTAAAAATTTAGGG...,1.55,CAAGATAGCGTTTAAAAATTTAGGGGTGTTAGGCTCAGCGT,552,None,None,None
6,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,300,300,740,+,.,context=AGGGGTGTTAGGCTCAGCGTAGAGTTTGCCAAGCTCTA...,5.36,AGGGGTGTTAGGCTCAGCGTAGAGTTTGCCAAGCTCTATGC,545,RCGDAD,695,RCGDAD
7,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m4C,309,309,116,+,.,coverage=550;context=AGGCTCAGCGTAGAGTTTGCCAAGC...,1.94,AGGCTCAGCGTAGAGTTTGCCAAGCTCTATGCATTCATTGA,550,None,100,None
8,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,modified_base,331,331,35,+,.,coverage=554;context=AGCTCTATGCATTCATTGATGATGA...,1.41,AGCTCTATGCATTCATTGATGATGATAGGGTTTTGCGTGGG,554,None,None,None
9,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,modified_base,341,341,31,+,.,coverage=488;context=ATTCATTGATGATGATAGGGTTTTG...,1.36,ATTCATTGATGATGATAGGGTTTTGCGTGGGCGTGAAGCCA,488,None,None,None
10,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m4C,352,352,41,+,.,coverage=547;context=ATGATAGGGTTTTGCGTGGGCGTGA...,1.51,ATGATAGGGTTTTGCGTGGGCGTGAAGCCAATTTCATACGC,547,None,14,None
11,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,361,361,26,+,.,coverage=538;context=TTTTGCGTGGGCGTGAAGCCAATTT...,1.36,TTTTGCGTGGGCGTGAAGCCAATTTCATACGCTCCTAAGCG,538,None,8,None
13,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,383,383,526,+,.,context=TTTCATACGCTCCTAAGCGTAAAATCGCCTTTTCCATG...,4.56,TTTCATACGCTCCTAAGCGTAAAATCGCCTTTTCCATGCTC,525,RCGDAD,494,RCGDAD
15,CP003904_1_Helicobacter_pylori26695_complete_g...,kinModCall,m6A,398,398,694,+,.,context=AGCGTAAAATCGCCTTTTCCATGCTCCCTAATCGCTTG...,4.80,AGCGTAAAATCGCCTTTTCCATGCTCCCTAATCGCTTGAAA,549,CATG,659,CATG


In [23]:
# go -1 from start methyl mark, and fastq only have + end, so - end motifs aren't marked in dataframe
# for example 255 is a m6A found in ctrl_ph7 with motif GANTC
sequence[250:259]

'AAAGAATCC'

In [24]:
# next thing to do is index everywhere with a + strand 
ctrl_ph7 = ctrl_ph7[ctrl_ph7['strand'] == '+']
ctrl_ph5 = ctrl_ph5[ctrl_ph5['strand'] == '+']

arss_ph7 = arss_ph7[arss_ph7['strand'] == '+']
arss_ph5 = arss_ph5[arss_ph5['strand'] == '+']

hsdm_ph7 = hsdm_ph7[hsdm_ph7['strand'] == '+']
hsdm_ph5 = hsdm_ph5[hsdm_ph5['strand'] == '+']

In [25]:
print(f"\nTotal motifs found: {len(cga)}")


Total motifs found: 609


In [15]:
#now, we're going to compare the start colum of ctrl_ph7 to cga

In [16]:
!pip install intervaltree


In [26]:
tree = IntervalTree(Interval(start, start + 1) for start in ctrl_ph7['start'])

def is_methylated(start, end, tree):
    return any(tree[start:end + 1])

cga['methylated in ctrl_ph7'] = cga.apply(lambda row: is_methylated(row['Start'], row['End'], tree), axis=1)

cga

,Start,End,Motif,methylated in ctrl_ph7
0,3680,3691,CGAAATGTATC,False
1,5752,5763,CGATCCCATTC,True
2,8759,8770,CGATGAGTTTC,False
3,10440,10451,CGATAACGATC,True
4,17458,17469,CGAAGCTTATC,True
...,...,...,...,...
604,1650155,1650166,CGATTTTCTTC,True
605,1650374,1650385,CGATTGTCCTC,False
606,1656882,1656893,CGAGCATTATC,False
607,1664006,1664017,CGAGGGCTATC,True


In [27]:
cga

,Start,End,Motif,methylated in ctrl_ph7
0,3680,3691,CGAAATGTATC,False
1,5752,5763,CGATCCCATTC,True
2,8759,8770,CGATGAGTTTC,False
3,10440,10451,CGATAACGATC,True
4,17458,17469,CGAAGCTTATC,True
...,...,...,...,...
604,1650155,1650166,CGATTTTCTTC,True
605,1650374,1650385,CGATTGTCCTC,False
606,1656882,1656893,CGAGCATTATC,False
607,1664006,1664017,CGAGGGCTATC,True


In [14]:
def create_interval_tree(df):
    return IntervalTree(Interval(start, start + 1) for start in df['start'])

def is_methylated(start, end, tree):
    return any(tree[start:end + 1])

tree_ctrl_ph7 = create_interval_tree(ctrl_ph7)
tree_ctrl_ph5 = create_interval_tree(ctrl_ph5)
tree_hsdm_ph7 = create_interval_tree(hsdm_ph7)
tree_hsdm_ph5 = create_interval_tree(hsdm_ph5)

cga['methylated in ctrl_ph7'] = cga.apply(lambda row: is_methylated(row['Start'], row['End'], tree_ctrl_ph7), axis=1)
cga['methylated in ctrl_ph5'] = cga.apply(lambda row: is_methylated(row['Start'], row['End'], tree_ctrl_ph5), axis=1)
cga['methylated in hsdm_ph7'] = cga.apply(lambda row: is_methylated(row['Start'], row['End'], tree_hsdm_ph7), axis=1)
cga['methylated in hsdm_ph5'] = cga.apply(lambda row: is_methylated(row['Start'], row['End'], tree_hsdm_ph5), axis=1)

cga

,Start,End,Motif,methylated in ctrl_ph7,methylated in ctrl_ph5,methylated in hsdm_ph7,methylated in hsdm_ph5
0,3680,3691,CGAAATGTATC,False,False,False,False
1,5752,5763,CGATCCCATTC,True,True,True,True
2,8759,8770,CGATGAGTTTC,False,False,False,False
3,10440,10451,CGATAACGATC,True,True,True,True
4,17458,17469,CGAAGCTTATC,True,False,False,True
...,...,...,...,...,...,...,...
604,1650155,1650166,CGATTTTCTTC,True,True,True,True
605,1650374,1650385,CGATTGTCCTC,False,False,False,False
606,1656882,1656893,CGAGCATTATC,False,False,False,False
607,1664006,1664017,CGAGGGCTATC,True,True,True,True


# CCANNNNNNTC

In [16]:
def find_motifs(sequence): # for CCANNNNNNTC
    pattern = 'CGRAT'  #equivalent to CCANNNNNNTC
    results = []

    for match in re.finditer(pattern, sequence):
        start = match.start()
        end = match.end()
        actual_motif = sequence[start:end]
        results.append({
            'Start': start,
            'End': end,
            'Motif': actual_motif
        })

    return pd.DataFrame(results)

# Assuming you've already fetched and stored your sequence as a string variable 'sequence'
cca = find_motifs(sequence)

In [17]:
def create_interval_tree(df):
    return IntervalTree(Interval(start, start + 1) for start in df['start'])

def is_methylated(start, end, tree):
    return any(tree[start:end + 1])

tree_ctrl_ph7 = create_interval_tree(ctrl_ph7)
tree_ctrl_ph5 = create_interval_tree(ctrl_ph5)
tree_hsdm_ph7 = create_interval_tree(hsdm_ph7)
tree_hsdm_ph5 = create_interval_tree(hsdm_ph5)

cca['methylated in ctrl_ph7'] = cca.apply(lambda row: is_methylated(row['Start'], row['End'], tree_ctrl_ph7), axis=1)
cca['methylated in ctrl_ph5'] = cca.apply(lambda row: is_methylated(row['Start'], row['End'], tree_ctrl_ph5), axis=1)
cca['methylated in hsdm_ph7'] = cca.apply(lambda row: is_methylated(row['Start'], row['End'], tree_hsdm_ph7), axis=1)
cca['methylated in hsdm_ph5'] = cca.apply(lambda row: is_methylated(row['Start'], row['End'], tree_hsdm_ph5), axis=1)

cca[1:100]

ValueError: Cannot set a DataFrame without columns to the column methylated in ctrl_ph7

## well, how can i not do rcgdad...
### this can be applied now to any motif with degenerate bases!

In [17]:
def find_motifs(sequence, motif):
    degenerate_bases = {
        'R': '[AG]',
        'Y': '[CT]',
        'S': '[GC]',
        'W': '[AT]',
        'K': '[GT]',
        'M': '[AC]',
        'B': '[CGT]',
        'D': '[AGT]',
        'H': '[ACT]',
        'V': '[ACG]',
        'N': '[ATCG]',
        'A': 'A',
        'C': 'C',
        'G': 'G',
        'T': 'T'
    }

    # Convert the motif into a regular expression pattern
    pattern = ''.join(degenerate_bases[base] for base in motif)
    results = []

    for match in re.finditer(pattern, sequence):
        start = match.start()
        end = match.end()
        actual_motif = sequence[start:end]
        results.append({
            'Start': start,
            'End': end,
            'Motif': actual_motif
        })

    return pd.DataFrame(results)

def create_interval_tree(df):
    return IntervalTree(Interval(start, start + 1) for start in df['start'])

def is_methylated(start, end, tree):
    return any(tree[start:end + 1])


tree_ctrl_ph7 = create_interval_tree(ctrl_ph7)
tree_ctrl_ph5 = create_interval_tree(ctrl_ph5)
tree_hsdm_ph7 = create_interval_tree(hsdm_ph7)
tree_hsdm_ph5 = create_interval_tree(hsdm_ph5)

In [18]:
motif = "CGRAT"
rcgdad = find_motifs(sequence, motif)
rcgdad

,Start,End,Motif
0,1911,1916,CGGAT
1,5849,5854,CGAAT
2,7984,7989,CGAAT
3,10361,10366,CGAAT
4,10901,10906,CGGAT
...,...,...,...
1654,1661716,1661721,CGGAT
1655,1663031,1663036,CGAAT
1656,1663729,1663734,CGGAT
1657,1664230,1664235,CGGAT


In [19]:
rcgdad['methylated in ctrl_ph7'] = rcgdad.apply(lambda row: is_methylated(row['Start'], row['End'], tree_ctrl_ph7), axis=1)
rcgdad['methylated in ctrl_ph5'] = rcgdad.apply(lambda row: is_methylated(row['Start'], row['End'], tree_ctrl_ph5), axis=1)
rcgdad['methylated in hsdm_ph7'] = rcgdad.apply(lambda row: is_methylated(row['Start'], row['End'], tree_hsdm_ph7), axis=1)
rcgdad['methylated in hsdm_ph5'] = rcgdad.apply(lambda row: is_methylated(row['Start'], row['End'], tree_hsdm_ph5), axis=1)

rcgdad

,Start,End,Motif,methylated in ctrl_ph7,methylated in ctrl_ph5,methylated in hsdm_ph7,methylated in hsdm_ph5
0,1911,1916,CGGAT,True,True,True,True
1,5849,5854,CGAAT,True,True,True,True
2,7984,7989,CGAAT,True,True,True,True
3,10361,10366,CGAAT,True,True,True,True
4,10901,10906,CGGAT,True,True,True,True
...,...,...,...,...,...,...,...
1654,1661716,1661721,CGGAT,True,True,True,True
1655,1663031,1663036,CGAAT,True,True,True,True
1656,1663729,1663734,CGGAT,True,True,True,True
1657,1664230,1664235,CGGAT,True,True,True,True


In [21]:
rcgdad[rcgdad['methylated in ctrl_ph5'] == False]

,Start,End,Motif,methylated in ctrl_ph7,methylated in ctrl_ph5,methylated in hsdm_ph7,methylated in hsdm_ph5
91,1555716,1555724,GCGCGCAT,True,False,True,False


In [26]:
rcgdad[rcgdad['methylated in ctrl_ph7'] == False]

,Start,End,Motif,methylated in ctrl_ph7,methylated in ctrl_ph5,methylated in hsdm_ph7,methylated in hsdm_ph5
217,1013883,1013889,CCGGAT,False,False,False,False


In [27]:
rcgdad[rcgdad['methylated in hsdm_ph7'] == False]

,Start,End,Motif,methylated in ctrl_ph7,methylated in ctrl_ph5,methylated in hsdm_ph7,methylated in hsdm_ph5
217,1013883,1013889,CCGGAT,False,False,False,False
